# **07 — Trực quan hóa kết quả mô hình phân loại bình luận tiêu cực tiếng Việt**

Notebook này đọc kết quả từ `output/model/` (Machine Learning) và `output/phobert/` (Deep Learning) để sinh toàn bộ **biểu đồ (figures)** phục vụ báo cáo.

**Danh sách biểu đồ:**

| # | Biểu đồ | File output | Yêu cầu báo cáo |
|---|---------|-------------|-----------------|
| 1 | Learning Curve — Logistic Regression | `lc_lr.png` | Kết quả thực nghiệm |
| 2 | Learning Curve — Linear SVM | `lc_svm.png` | Kết quả thực nghiệm |
| 3 | Learning Curve — Random Forest | `lc_rf.png` | Kết quả thực nghiệm |
| 4 | SGD Epoch Curves (Loss & Accuracy) | `sgd_epoch_curves.png` | Cấu hình huấn luyện |
| 5 | PhoBERT Training Curves (Loss & F1) | `phobert_curves.png` | Cấu hình huấn luyện (DL) |
| 6 | Confusion Matrix — tất cả 6 mô hình ML | `cm_*.png` | Đánh giá trên tập Test |
| 7 | Model Comparison Bar Chart (bao gồm PhoBERT) | `model_comparison.png` | So sánh hiệu năng |
| 8 | Per-class F1 Comparison (bao gồm PhoBERT) | `f1_per_class.png` | Đánh giá trên tập Test |
| 9 | Bảng tổng hợp Metrics (bao gồm PhoBERT) | `metrics_table.png` | Đánh giá trên tập Test |
| 10| Phân bố lỗi dự đoán sai (Voting Ensemble) | `error_distribution.png` | Phân tích lỗi |
| 11| Pipeline Architecture Diagram | `arch_pipeline.png` | Kiến trúc mô hình |

Tất cả figures được lưu vào `reports/report_2/figures/`.

---
## **Phần 0: Import thư viện**

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

_cwd = Path(".").resolve()
if (_cwd / "data").exists():
    ROOT_DIR = _cwd
elif (_cwd.parent / "data").exists():
    ROOT_DIR = _cwd.parent
else:
    raise FileNotFoundError("Không tìm thấy thư mục 'data/'.")

MODEL_DIR = ROOT_DIR / "output" / "model"
PHOBERT_DIR = ROOT_DIR / "output" / "phobert"
SAVE_DIR = ROOT_DIR / "reports" / "report_2" / "figures"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

LABEL_NAMES_VI = ["Sạch (CLEAN)", "Xúc phạm (OFFENSIVE)", "Thù ghét (HATE)"]
CLASS_KEYS = ["CLEAN", "OFFENSIVE", "HATE"]

MODEL_ORDER = [
    "LogisticRegression",
    "MultinomialNB",
    "LinearSVC",
    "RandomForest",
    "SGDClassifier",
    "VotingEnsemble",
    "PhoBERT"
]
DISPLAY_NAMES = ["LR", "NB", "SVM", "RF", "SGD", "Voting", "PhoBERT"]

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_style("whitegrid")

---
## **Phần 1: Đọc dữ liệu kết quả**

Đọc kết quả ML và DL (PhoBERT) rồi gộp chung lại để tiện so sánh.

In [2]:
def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

results = load_json(MODEL_DIR / "all_results.json")
learning_curves = load_json(MODEL_DIR / "learning_curve_data.json")
sgd_epochs = load_json(MODEL_DIR / "sgd_epoch_curve.json")
error_analysis = load_json(MODEL_DIR / "error_analysis.json")

# Nạp kết quả PhoBERT
phobert_data = load_json(PHOBERT_DIR / "phobert_results.json")
results["models"]["PhoBERT"] = {
    "dev": {
        "accuracy": phobert_data["epoch_metrics"]["val_accuracy"][-1],
        "f1_weighted": phobert_data["epoch_metrics"]["val_f1_weighted"][-1]
    },
    "test": phobert_data["test_metrics"]
}

print(f"Đã nạp kết quả cho {len(results['models'])} mô hình.")

Đã nạp kết quả cho 7 mô hình.


## **Phần 2: Learning Curves**

*Lưu ý: Chỉ có Logistic Regression, Linear SVM và Random Forest có Learning Curve vì quá trình tính toán rất tốn thời gian. Các mô hình khác (NB, SGD, Voting, PhoBERT) không được chạy Learning Curve.*

## **Phần 2: Learning Curve — Logistic Regression**

In [3]:
data = learning_curves["LogisticRegression"]
sizes = np.array(data["train_sizes"])
train_mean = np.array(data["train_mean"])
train_std = np.array(data["train_std"])
val_mean = np.array(data["val_mean"])
val_std = np.array(data["val_std"])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, train_mean, "o-", label="Train", color="#1f77b4", linewidth=2)
ax.fill_between(sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="#1f77b4")
ax.plot(sizes, val_mean, "s-", label="Validation", color="#ff7f0e", linewidth=2)
ax.fill_between(sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color="#ff7f0e")
ax.set_xlabel("Kích thước tập huấn luyện", fontsize=12)
ax.set_ylabel("F1-weighted", fontsize=12)
ax.set_title("Learning Curve — Logistic Regression", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(SAVE_DIR / "lc_lr.png", dpi=150, bbox_inches="tight")
plt.close()

## **Phần 2: Learning Curve — Linear SVM**

In [4]:
data = learning_curves["LinearSVC"]
sizes = np.array(data["train_sizes"])
train_mean = np.array(data["train_mean"])
train_std = np.array(data["train_std"])
val_mean = np.array(data["val_mean"])
val_std = np.array(data["val_std"])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, train_mean, "o-", label="Train", color="#1f77b4", linewidth=2)
ax.fill_between(sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="#1f77b4")
ax.plot(sizes, val_mean, "s-", label="Validation", color="#ff7f0e", linewidth=2)
ax.fill_between(sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color="#ff7f0e")
ax.set_xlabel("Kích thước tập huấn luyện", fontsize=12)
ax.set_ylabel("F1-weighted", fontsize=12)
ax.set_title("Learning Curve — Linear SVM", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(SAVE_DIR / "lc_svm.png", dpi=150, bbox_inches="tight")
plt.close()

## **Phần 2: Learning Curve — Random Forest**

In [5]:
data = learning_curves["RandomForest"]
sizes = np.array(data["train_sizes"])
train_mean = np.array(data["train_mean"])
train_std = np.array(data["train_std"])
val_mean = np.array(data["val_mean"])
val_std = np.array(data["val_std"])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, train_mean, "o-", label="Train", color="#1f77b4", linewidth=2)
ax.fill_between(sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="#1f77b4")
ax.plot(sizes, val_mean, "s-", label="Validation", color="#ff7f0e", linewidth=2)
ax.fill_between(sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color="#ff7f0e")
ax.set_xlabel("Kích thước tập huấn luyện", fontsize=12)
ax.set_ylabel("F1-weighted", fontsize=12)
ax.set_title("Learning Curve — Random Forest", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(SAVE_DIR / "lc_rf.png", dpi=150, bbox_inches="tight")
plt.close()

## **Phần 3: SGD Epoch Curves (Loss & Accuracy)**

In [6]:
epochs = np.array(sgd_epochs["epoch"])
train_loss = np.array(sgd_epochs["train_loss"])
val_loss = np.array(sgd_epochs["val_loss"])
train_acc = np.array(sgd_epochs["train_acc"])
val_acc = np.array(sgd_epochs["val_acc"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, train_loss, "o-", label="Train", color="#1f77b4")
axes[0].plot(epochs, val_loss, "s-", label="Validation", color="#ff7f0e")
axes[0].set_title("SGD — Log Loss theo Epoch", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, train_acc, "o-", label="Train", color="#1f77b4")
axes[1].plot(epochs, val_acc, "s-", label="Validation", color="#ff7f0e")
axes[1].set_title("SGD — Accuracy theo Epoch", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(SAVE_DIR / "sgd_epoch_curves.png", dpi=150, bbox_inches="tight")
plt.close()

## **Phần 3B: PhoBERT Training Curves**

In [7]:
epochs = np.array(phobert_data["epoch_metrics"]["epoch"])
train_loss = np.array(phobert_data["epoch_metrics"]["train_loss"])
val_loss = np.array(phobert_data["epoch_metrics"]["val_loss"])
val_f1 = np.array(phobert_data["epoch_metrics"]["val_f1_macro"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs, train_loss, "o-", label="Train Loss", color="#1f77b4", linewidth=2)
axes[0].plot(epochs, val_loss, "s-", label="Validation Loss", color="#ff7f0e", linewidth=2)
axes[0].set_title("PhoBERT — Loss theo Epoch", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("CrossEntropy Loss")
axes[0].set_xticks(epochs)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1
axes[1].plot(epochs, val_f1, "d-", label="Val F1-Macro", color="#2ca02c", linewidth=2)
axes[1].set_title("PhoBERT — Validation F1-Macro theo Epoch", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1-Macro")
axes[1].set_xticks(epochs)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(SAVE_DIR / "phobert_curves.png", dpi=150, bbox_inches="tight")
plt.close()

## **Phần 4: Confusion Matrix**

*Lưu ý: PhoBERT không được vẽ Confusion Matrix ở đây vì output gốc của PhoBERT quá nặng và chỉ in ra Classification Report, không lưu lại dữ liệu dự đoán chi tiết.*

In [8]:
for idx, model_key in enumerate(MODEL_ORDER[:-1]):
    fig, ax = plt.subplots(figsize=(7, 5))
    cm = np.array(results["models"][model_key]["confusion_matrix_test"])
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
                xticklabels=LABEL_NAMES_VI, yticklabels=LABEL_NAMES_VI, ax=ax)
    ax.set_ylabel("Thực tế", fontsize=12, fontweight="bold")
    ax.set_xlabel("Dự đoán", fontsize=12, fontweight="bold")
    ax.set_title(f"Confusion Matrix — {DISPLAY_NAMES[idx]}", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(SAVE_DIR / f"cm_{model_key.lower()}.png", dpi=150, bbox_inches="tight")
    plt.close()


## **Phần 5: So sánh hiệu năng các mô hình (Bao gồm PhoBERT)**

In [9]:
dev_acc = [results["models"][m]["dev"]["accuracy"] for m in MODEL_ORDER]
dev_f1 = [results["models"][m]["dev"]["f1_weighted"] for m in MODEL_ORDER]
test_acc = [results["models"][m]["test"]["accuracy"] for m in MODEL_ORDER]
test_f1 = [results["models"][m]["test"]["f1_weighted"] for m in MODEL_ORDER]

x = np.arange(len(MODEL_ORDER))
width = 0.35
best_idx = int(np.argmax(test_f1)) # Sẽ tự động chọn PhoBERT hoặc mô hình tốt nhất

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

colors_acc = ["#8da0cb"] * len(MODEL_ORDER)
colors_acc[best_idx] = "#fc8d62"
colors_f1 = ["#66c2a5"] * len(MODEL_ORDER)
colors_f1[-1] = "#e74c3c" # Highlight PhoBERT

axes[0].bar(x - width / 2, dev_acc, width, label="Accuracy", color=colors_acc, edgecolor="white")
axes[0].bar(x + width / 2, dev_f1, width, label="F1-weighted", color=colors_f1, edgecolor="white")
axes[0].set_title("Dev Set Metrics", fontsize=13, fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(DISPLAY_NAMES, fontsize=11)
axes[0].legend()
axes[0].set_ylim(0.5, 1.0)
axes[0].grid(True, axis="y", alpha=0.3)

axes[1].bar(x - width / 2, test_acc, width, label="Accuracy", color=colors_acc, edgecolor="white")
axes[1].bar(x + width / 2, test_f1, width, label="F1-weighted", color=colors_f1, edgecolor="white")
axes[1].set_title("Test Set Metrics", fontsize=13, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(DISPLAY_NAMES, fontsize=11)
axes[1].legend()
axes[1].grid(True, axis="y", alpha=0.3)

fig.suptitle("So sánh hiệu năng các mô hình", fontsize=15, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(SAVE_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.close()

## **Phần 6: Per-class F1 (Bao gồm PhoBERT)**

In [10]:
f1_by_model = [results["models"][m]["test"]["per_class_f1"] for m in MODEL_ORDER]

x = np.arange(len(MODEL_ORDER))
width = 0.25
colors = ["#2ecc71", "#e74c3c", "#9b59b6"]

fig, ax = plt.subplots(figsize=(14, 6))

for idx, class_key in enumerate(CLASS_KEYS):
    values = [f1[class_key] for f1 in f1_by_model]
    bars = ax.bar(x + (idx - 1) * width, values, width, label=LABEL_NAMES_VI[idx], color=colors[idx], edgecolor="white")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(DISPLAY_NAMES, fontsize=12)
ax.set_ylabel("F1-score")
ax.set_title("F1-Score theo từng lớp trên Test Set", fontsize=14, fontweight="bold")
ax.legend(loc="lower right")
ax.grid(True, axis="y", alpha=0.3)
ax.set_ylim(0, 1.05)

fig.tight_layout()
fig.savefig(SAVE_DIR / "f1_per_class.png", dpi=150, bbox_inches="tight")
plt.close()

## **Phần 7: Bảng tổng hợp Metrics**

In [11]:
rows = []
for m_key, d_name in zip(MODEL_ORDER, DISPLAY_NAMES):
    m = results["models"][m_key]
    rows.append({
        "Mô hình": d_name,
        "Dev Acc": f"{m['dev']['accuracy']:.4f}",
        "Dev F1-w": f"{m['dev']['f1_weighted']:.4f}",
        "Test Acc": f"{m['test']['accuracy']:.4f}",
        "Test Prec-w": f"{m['test']['precision_weighted']:.4f}",
        "Test Rec-w": f"{m['test']['recall_weighted']:.4f}",
        "Test F1-w": f"{m['test']['f1_weighted']:.4f}",
        "Test F1-macro": f"{m['test']['f1_macro']:.4f}",
    })

df_metrics = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(16, 4))
ax.axis("off")
table = ax.table(cellText=df_metrics.values, colLabels=df_metrics.columns, loc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.6)

for j in range(len(df_metrics.columns)):
    table[0, j].set_facecolor("#4472C4")
    table[0, j].set_text_props(color="white", fontweight="bold")

best_row_idx = df_metrics["Test F1-macro"].astype(float).idxmax() + 1
for j in range(len(df_metrics.columns)):
    table[best_row_idx, j].set_facecolor("#E2EFDA")

ax.set_title("Bảng tổng hợp các độ đo đánh giá", fontsize=13, fontweight="bold", pad=20)
fig.tight_layout()
fig.savefig(SAVE_DIR / "metrics_table.png", dpi=150, bbox_inches="tight")
plt.close()

## **Phần 8: Phân tích Overfitting (Tất cả mô hình)**

In [12]:
# Thay vì dùng CV_score (Voting và PhoBERT không có), ta sẽ dùng Dev F1-weighted để so sánh với Test F1-weighted
dev_scores = [results["models"][m]["dev"]["f1_weighted"] for m in MODEL_ORDER]
test_f1_w = [results["models"][m]["test"]["f1_weighted"] for m in MODEL_ORDER]
gaps = [dev - test for dev, test in zip(dev_scores, test_f1_w)]

x = np.arange(len(MODEL_ORDER))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width / 2, dev_scores, width, label="Dev F1-w", color="#3498db")
ax.bar(x + width / 2, test_f1_w, width, label="Test F1-w", color="#e74c3c")

for i, gap in enumerate(gaps):
    ax.annotate(f"Gap\n{gap:.3f}", xy=(x[i], min(dev_scores[i], test_f1_w[i])),
                xytext=(x[i], max(dev_scores[i], test_f1_w[i]) + 0.02),
                fontsize=9, ha="center", arrowprops=dict(arrowstyle="<->", color="#666"))

ax.set_xticks(x)
ax.set_xticklabels(DISPLAY_NAMES, fontsize=12)
ax.set_title("Overfitting: Dev F1 vs Test F1", fontsize=14, fontweight="bold")
ax.legend()
fig.savefig(SAVE_DIR / "overfitting_analysis.png", dpi=150, bbox_inches="tight")
plt.close()

## **Phần 9: Phân bố lỗi (Voting Ensemble)**

In [13]:
confusion_counts = error_analysis["confusion_counts"]
label_map = {"0": "CLEAN", "1": "OFFENSIVE", "2": "HATE"}

error_labels = []
error_values = []
for k, v in sorted(confusion_counts.items(), key=lambda x: -x[1]):
    t, p = k.split("->")
    error_labels.append(f"{label_map[t]} -> {label_map[p]}")
    error_values.append(v)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(error_labels)), error_values, color="#e74c3c")
for bar, val in zip(bars, error_values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            f"{val}", va="center", fontweight="bold")
ax.set_yticks(range(len(error_labels)))
ax.set_yticklabels(error_labels)
ax.set_title("Phân bố lỗi khi dự đoán — Voting Ensemble", fontsize=14, fontweight="bold")
ax.invert_yaxis()
fig.savefig(SAVE_DIR / "error_distribution.png", dpi=150, bbox_inches="tight")
plt.close()

## **Phần 10: Sơ đồ kiến trúc (Mở rộng ML & DL)**

In [14]:
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(16, 5))
ax.axis("off")

# Core Data processing
bbox_raw = dict(boxstyle="round,pad=0.5", fc="#e8f4fd", ec="#2980b9")
bbox_nlp = dict(boxstyle="round,pad=0.5", fc="#fef9e7", ec="#f39c12")
ax.text(0.1, 0.5, "Văn bản thô", ha="center", va="center", bbox=bbox_raw, zorder=3)
ax.text(0.3, 0.5, "Tiền xử lý NLP\n(Clean, Teencode)", ha="center", va="center", bbox=bbox_nlp, zorder=3)

# ML Branch
bbox_ml1 = dict(boxstyle="round,pad=0.5", fc="#eafaf1", ec="#27ae60")
bbox_ml2 = dict(boxstyle="round,pad=0.5", fc="#f4ecf7", ec="#8e44ad")
ax.text(0.5, 0.8, "TF-IDF + SMOTE\n(Feature Eng.)", ha="center", va="center", bbox=bbox_ml1, zorder=3)
ax.text(0.7, 0.8, "Machine Learning\n(LR, SVM, RF, Voting)", ha="center", va="center", bbox=bbox_ml2, zorder=3)

# DL Branch
bbox_dl1 = dict(boxstyle="round,pad=0.5", fc="#eafaf1", ec="#16a085")
bbox_dl2 = dict(boxstyle="round,pad=0.5", fc="#f4ecf7", ec="#9b59b6")
ax.text(0.5, 0.2, "PhoBERT Tokenizer\n(max_length=256)", ha="center", va="center", bbox=bbox_dl1, zorder=3)
ax.text(0.7, 0.2, "Deep Learning\n(PhoBERT + Weighted Loss)", ha="center", va="center", bbox=bbox_dl2, zorder=3)

# Output
bbox_out = dict(boxstyle="round,pad=0.5", fc="#fdedec", ec="#e74c3c")
ax.text(0.9, 0.5, "Nhãn Đầu Ra\n0/1/2", ha="center", va="center", bbox=bbox_out, zorder=3)

# Add explicit arrows using FancyArrowPatch
arrow_style = "simple,tail_width=0.05,head_width=0.4,head_length=0.4"
kw = dict(arrowstyle=arrow_style, color="black", zorder=1)

# Raw to NLP
ax.add_patch(patches.FancyArrowPatch((0.15, 0.5), (0.23, 0.5), **kw))

# NLP to ML Branch
ax.add_patch(patches.FancyArrowPatch((0.3, 0.55), (0.3, 0.8), connectionstyle="arc3,rad=0", **kw))
ax.add_patch(patches.FancyArrowPatch((0.3, 0.8), (0.42, 0.8), **kw))
ax.add_patch(patches.FancyArrowPatch((0.58, 0.8), (0.6, 0.8), **kw))
ax.add_patch(patches.FancyArrowPatch((0.8, 0.8), (0.9, 0.8), connectionstyle="arc3,rad=0", **kw))
ax.add_patch(patches.FancyArrowPatch((0.9, 0.8), (0.9, 0.55), **kw))

# NLP to DL Branch
ax.add_patch(patches.FancyArrowPatch((0.3, 0.45), (0.3, 0.2), connectionstyle="arc3,rad=0", **kw))
ax.add_patch(patches.FancyArrowPatch((0.3, 0.2), (0.42, 0.2), **kw))
ax.add_patch(patches.FancyArrowPatch((0.58, 0.2), (0.6, 0.2), **kw))
ax.add_patch(patches.FancyArrowPatch((0.8, 0.2), (0.9, 0.2), connectionstyle="arc3,rad=0", **kw))
ax.add_patch(patches.FancyArrowPatch((0.9, 0.2), (0.9, 0.45), **kw))

ax.set_title("So do kien truc: Machine Learning vs Deep Learning", fontsize=15, fontweight="bold")
fig.savefig(SAVE_DIR / "arch_pipeline.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved all figures including PhoBERT!")

Saved all figures including PhoBERT!
